# 07 · Smart money: insiders, 13F holders and investment advisers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/valuein/blob/main/examples/notebooks/07_smart_money.ipynb)

**What you will learn**

- What the smart-money tables hold: insider transactions (Forms 3, 4, 5 and 144),
  institutional holdings (13F), 5%+ blockholders (13D and 13G) and investment-adviser profiles
  (Form ADV).
- When each filing becomes public, and why that date is the one a backtest must use.
- How to call the SDK methods, and how a plan check fails fast with a clear error.

**Plan required: Institutional.** On any other plan, including the free sample, this notebook
still runs: each data call stops with a `ValueinPlanError` *before* any request is sent, and
the notebook prints that message. Run it with an Institutional key in `VALUEIN_API_KEY` to see
the data.

In [1]:
%pip install -q valuein-sdk

Note: you may need to restart the kernel to use updated packages.


## 1. Check your plan, read the columns for free

The schema of every table comes with the manifest, so you can see the columns of a gated table
on any plan.

In [2]:
from valuein_sdk import ValueinClient, ValueinPlanError

client = ValueinClient()
print("plan:", client.plan)

for table in ["insider_transaction", "institutional_holding", "insider_ownership", "investment_adviser"]:
    columns = list(client.get_schema(table))
    print(f"{table:<22} {len(columns):>3} columns: {', '.join(columns[:8])}, ...")


Upgrade access:
  Register (free)  → Benchmark: full history for every S&P 500 constituent
  Add API token    → Full US equities universe + complete history

Get your token: https://valuein.biz
  client = ValueinClient()


plan: sample
insider_transaction     25 columns: id, accession_id, transaction_line_no, entity_id, insider_party_id, form_type, transaction_type, transaction_date, ...
institutional_holding   18 columns: id, accession_id, filer_cik, filer_name, period_end, share_class_figi, subject_entity_id, name_of_issuer, ...
insider_ownership       17 columns: id, accession_id, filer_cik, filer_party_id, filer_name, filer_type, subject_entity_id, schedule_type, ...
investment_adviser      61 columns: id, crd_number, filing_date, sec_number, business_name, legal_name, sec_region, umbrella_registration, ...


## 2. Insider transactions

Officers, directors and 10% owners report trades on Form 4 within two business days.
`transaction_code` `P` is an open-market purchase, `S` an open-market sale. Rows are filtered
by `accepted_at`, the moment the filing became public, not the transaction date.

In [3]:
try:
    insiders = client.insider_transactions("NVDA", transaction_codes=["P", "S"], limit=50)
    print(insiders[["transaction_date", "insider_name", "insider_role", "transaction_code",
                    "shares", "price_per_share", "accepted_at"]].head(10))
except ValueinPlanError as exc:
    print("ValueinPlanError:", exc)

ValueinPlanError: 'insider_transaction' requires the Institutional (full) plan; your plan is 'sample'. Upgrade at https://valuein.biz/pricing — no request was spent on this call.


## 3. Who owns the stock: 13F holdings

Managers with more than $100 million in US equities report holdings each quarter on Form 13F,
up to 45 days after quarter end. A 13F position is therefore known only at the filing's
`accepted_at`, weeks after the `period_end` it describes. Securities are identified by FIGI
(`share_class_figi`) and by issuer CIK (`subject_entity_id`).

In [4]:
try:
    holders = client.institutional_holdings(issuer="NVDA", limit=200)
    latest = holders[holders["period_end"] == holders["period_end"].max()]
    print(latest[["filer_name", "shares", "market_value_usd", "period_end", "accepted_at"]].head(10))
except ValueinPlanError as exc:
    print("ValueinPlanError:", exc)

ValueinPlanError: 'institutional_holding' requires the Institutional (full) plan; your plan is 'sample'. Upgrade at https://valuein.biz/pricing — no request was spent on this call.


## 4. What one manager holds

Pass `filer_cik` instead of `issuer` to list a manager's own portfolio. Berkshire Hathaway's
CIK is 0001067983.

In [5]:
try:
    portfolio = client.institutional_holdings(filer_cik="0001067983", limit=200)
    latest = portfolio[portfolio["period_end"] == portfolio["period_end"].max()]
    print(latest[["name_of_issuer", "share_class_figi", "shares", "market_value_usd"]].head(10))
except ValueinPlanError as exc:
    print("ValueinPlanError:", exc)

ValueinPlanError: 'institutional_holding' requires the Institutional (full) plan; your plan is 'sample'. Upgrade at https://valuein.biz/pricing — no request was spent on this call.


## 5. Investment advisers (Form ADV)

Advisers file Form ADV with the SEC's IARD system, keyed by CRD number, not CIK. Three facts
matter when you use these numbers:

- `cik` is a nullable link: many advisers never file on EDGAR.
- Items 5.A to 5.F (employees, clients) are absent for Exempt Reporting Advisers, because they
  file under a different obligation. Absent is not zero.
- `raum_total` (regulatory assets under management) counts sub-advised assets at both the
  adviser and the sub-adviser. Compare it firm by firm; never sum it across firms.

In [6]:
try:
    advisers = client.advisers(name_contains="bridgewater", limit=5)
    print(advisers[["crd_number", "business_name", "raum_total", "registration_status", "filing_date"]])
except ValueinPlanError as exc:
    print("ValueinPlanError:", exc)

ValueinPlanError: 'investment_adviser' requires the Institutional (full) plan; your plan is 'sample'. Upgrade at https://valuein.biz/pricing — no request was spent on this call.


## 6. Ready-made templates

Four SQL templates cover the common questions. They run only on the Institutional plan, so this
cell checks the plan first instead of letting the query fail.

| Template | Question |
|---|---|
| `insider_buys` | open-market purchases for a ticker in the last N days |
| `top_institutional_holders` | the largest 13F holders of a ticker |
| `manager_portfolio` | one manager's positions with quarter-over-quarter changes |
| `blockholders` | 13D and 13G filers, flagging a switch from passive (13G) to active (13D) |

In [7]:
if client.plan == "full":
    print(client.run_template("insider_buys", ticker="NVDA", lookback_days=180, min_shares=1000).head())
    print(client.run_template("top_institutional_holders", ticker="NVDA", top_n=10))
    print(client.run_template("blockholders", ticker="NVDA", lookback_days=365).head())
else:
    print(f"Plan '{client.plan}': the smart-money templates need the Institutional plan.\n"
          "Plans and prices: https://valuein.biz/pricing")

Plan 'sample': the smart-money templates need the Institutional plan.
Plans and prices: https://valuein.biz/pricing


In [8]:
client.close()

## Next steps

- **[08 · Verify a number](08_verify_a_number.ipynb)**: trace any value to the SEC filing it
  came from.
- Script version: [`python/smart_money_screen.py`](../python/smart_money_screen.py).